# SPY 0DTE Long Iron Condor — Colab Runner

Run the backtest from a phone or any browser. You need a Polygon Options Advanced/Developer key.

**Strategy in one line:** every minute from 9:50 ET, look at RSI on 1-min SPY; the first time it crosses above the upper threshold or below the lower threshold inside the entry window, buy a long iron condor and exit on the first of (profit target, stop loss, time stop).

Cells below: install → key → sweep → results → timing analysis.

## 1. Clone the repo (fast — under 30 s on a cold runtime)
Skips the slow `pip install -e .` editable build and just adds `src/` to PYTHONPATH. Only `exchange_calendars` needs installing — everything else is pre-shipped in Colab.

In [ ]:
import os, sys
REPO_DIR = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO_DIR}

%cd {REPO_DIR}
!git pull --quiet

# Install only the deps Colab does NOT ship with by default.
!pip install -q exchange_calendars

# Put src on PYTHONPATH instead of running the slow editable install.
SRC = os.path.join(REPO_DIR, 'src')
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

print('OK')

## 2. Paste your Polygon API key
Uses `getpass` so the key is not saved in the notebook. If the cell is re-run, paste again.

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Run the sweep (240 configs, ~3 min with warm cache)
Defaults: 30 days, `fixed_1.0x3` strike rule, RSI 14, four RSI thresholds (70/30, 73/27, 75/25, 80/20), all profit targets, all stop losses, all entry cutoffs.

First run on a fresh Colab runtime will be slower (~10–15 min) because the Polygon data cache is empty and has to be filled. Subsequent runs are fast.

In [ ]:
!python -m iron_condor.cli --sweep

## 4. Look at the top configs

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

## 5. Timing analysis — how long do winners win vs losers lose?
Use this to decide where an early time-stop should sit. Look at the **profit p75 / p90 / max** — that's how long winners typically need to develop. Set the early time-stop to be *longer* than the winner p90, so we don't kill real winners. The `time_stop` and `stop` distributions tell us how the losing trades behave.

In [ ]:
from iron_condor.metrics import analyze_timing
trades = pd.read_csv('results/sweep_trades.csv')

print('=== Timing across all configs ===')
print(analyze_timing(trades))

In [ ]:
# Per top-5 configs — see if timing varies by config
top5 = summary.head(5)['config'].tolist()
for cfg in top5:
    print(f'\n--- {cfg} ---')
    print(analyze_timing(trades[trades['config'] == cfg]))

In [ ]:
# Per RSI threshold — does a higher threshold change timing?
trades['rsi_thresh'] = trades['config'].str.extract(r'rsi\d+_(\d+-\d+)')
for thr in trades['rsi_thresh'].dropna().unique():
    print(f'\n--- RSI threshold {thr} ---')
    print(analyze_timing(trades[trades['rsi_thresh'] == thr]))

## 6. (Optional) Persist the Polygon cache to Google Drive
Colab kills idle sessions after ~90 min and the local disk is wiped. Mount Drive so option bars are cached across sessions. Only useful if you plan to run multiple sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
PERSIST = '/content/drive/MyDrive/iron-condor-cache'
LOCAL   = '/content/iron-condor/data/cache'
os.makedirs(PERSIST, exist_ok=True)
if os.path.islink(LOCAL):
    pass
else:
    if os.path.isdir(LOCAL):
        shutil.rmtree(LOCAL)
    os.symlink(PERSIST, LOCAL)
print('cache now lives at', PERSIST)